In [ ]:
import time
import re
import random
import logging
import datetime
import argparse
import ast
import pandas as pd
import numpy as np
from huggingface_hub import login
from collections import Counter


In [ ]:
full_train_df = pd.read_csv("/cluster/work/projects/ec403/ec-michechi/Project_M/data/landmark_evo_train.csv", na_values=['', 'None', 'NaN', 'na', 'nan']).fillna('')

In [ ]:
filtered_datasets = []
for dataset in [full_train_df]:
    visit_counts = dataset['subject_id'].value_counts()
    selected_patients = visit_counts[visit_counts == 3].index
    dataset_selected = dataset[dataset['subject_id'].isin(selected_patients)].copy()
    filtered_datasets.append(dataset_selected)

train_df_selected = filtered_datasets[0]

In [ ]:
train_df_selected.head(3)

In [ ]:
def compact_narrative_prompt(row):
    narrative = f"You are a Doctor.\nWhat is the probability of death in the next 90 days for {row['age_at_landmark']}-year-old {row['gender']} patient?\n"
    current_visit = row['landmark_visit']
    type = row['admission_category']
    narrative += f"Visit number {current_visit} - {type} \n"

    if row['days_since_previous_visit'] != -1:
        narrative += f"Last visit happened {row['days_since_previous_visit']} days ago."

    # --- Diagnosi ---
    narrative += "\nDIAGNOSIS HISTORY:"
    diag_per_visit = ast.literal_eval(row['diag_per_visit'])
    
    # Conta frequenze
    all_diags = []
    for diags in diag_per_visit.values():
        all_diags.extend(diags)
    diag_counts = Counter(all_diags)

    # Diagnosi croniche (almeno 2 visite)
    chronic_diags = [d for d, c in diag_counts.items() if c >= 2]

    # Diagnosi nuove solo in questa visita
    current_diags = diag_per_visit[int(current_visit)]
    new_diags = [d for d in current_diags if diag_counts[d] == 1]

    if chronic_diags:
        narrative += f"\nChronic diagnoses: {', '.join(chronic_diags)}."
    if new_diags:
        narrative += f"\nNew diagnoses in this visit: {', '.join(new_diags)}."
    if not chronic_diags and not new_diags:
        narrative += "\nNo diagnoses recorded."

    # --- Farmaci ---
    narrative += "\nPRESCRIPTIONS HISTORY:"
    meds_per_visit = ast.literal_eval(row['meds_per_visit'])
    
    all_meds = []
    for meds in meds_per_visit.values():
        all_meds.extend(meds)
    med_counts = Counter(all_meds)

    chronic_meds = [m for m, c in med_counts.items() if c >= 2]
    current_meds = meds_per_visit[int(current_visit)]
    new_meds = [m for m in current_meds if med_counts[m] == 1]

    if chronic_meds:
        narrative += f"\nChronic medications: {', '.join(chronic_meds)}."
    if new_meds:
        narrative += f"\nNew medications in this visit: {', '.join(new_meds)}."
    if not chronic_meds and not new_meds:
        narrative += "\nNo medications recorded."

    # --- Procedure ---
    narrative += "\nPROCEDURES HISTORY:"
    proc_per_visit = ast.literal_eval(row['proc_per_visit'])

    all_proc = []
    for procs in proc_per_visit.values():
        all_proc.extend(procs)
    proc_counts = Counter(all_proc)

    chronic_proc = [p for p, c in proc_counts.items() if c >= 2]
    current_proc = proc_per_visit[int(current_visit)]
    new_proc = [p for p in current_proc if proc_counts[p] == 1]

    if chronic_proc:
        narrative += f"\nChronic procedures: {', '.join(chronic_proc)}."
    if new_proc:
        narrative += f"\nNew procedures in this visit: {', '.join(new_proc)}."
    if not chronic_proc and not new_proc:
        narrative += "\nNo procedures recorded."

    return narrative


In [ ]:
def naive_narrative_prompt(row):
    narrative = f"Patient is a {row['age_at_landmark']}-year-old {row['gender']}."
    narrative += f" This is the {row['num_total_visits']} visit."
    if row['days_since_previous_visit'] != -1:
        narrative += f" The last visit happened {row['days_since_previous_visit']} days ago."
    if pd.notna(row['diag_text']) and row['diag_text'].strip():
        narrative += f" Medical history includes: {row['diag_text']}."
    if pd.notna(row['med_text']) and row['med_text'].strip():
        narrative += f" Current medications are: {row['med_text']}."
    if pd.notna(row['proc_text']) and row['proc_text'].strip():
        narrative += f" Procedures performed: {row['proc_text']}."
    # Add explicit prediction question
    narrative += " Based on this information, what is the probability of mortality within 90 days?"
    return narrative

In [ ]:
print(compact_narrative_prompt(train_df_selected.iloc[1]))

In [ ]:
print(naive_narrative_prompt(train_df_selected.iloc[1]))